# 0. Import

A questo punto è stata effettuata la PCA delle features, con encoding sull'angolo Y.  
*(Avviare il codice relativo a Feature Extraction, PCA e la prima parte di Quantum Baseline dei colleghi, in particolare la parte relativa a Angle Scaling.)*

---

## 📌 Punto C3 – Topologie Ring e Ladder

Questo notebook approfondisce le topologie:

| Topologia | Supporto in Qiskit |
|-----------|--------------------|
| **Ring** | ✅ Nativa |
| **Ladder** | ❌ Non nativa (implementata manualmente) |

**Base utilizzata:** `RealAmplitudesShallow`

---

## ⚙️ Ottimizzatore

**SPSA custom** (fornito dal Tutor Cascone)

---

## ⚠️ Nota sugli osservabili `nn_pair`

> Nelle topologie **ring** e **ladder**, il numero di osservabili `nn_pair` è **superiore** rispetto alla topologia **linear**.  
> Verificarne l'impatto durante l'analisi.

## Nota sulla topologia `Circular`

> Nella topologia ad anello, il l'entaglement avviene prima tra il qubit 7 e qubit 0, che potrebbe risultare in un errore strutturale, dato il funzionamento della PCA e di conseguenza la natura delle feature compresse. Ci ritroveremo che il qubit con 'meno importanza' sporchi quella con piu' importanza

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
import math
import json
import time
from time import perf_counter
from typing import Any, Callable

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score
from torchmetrics.classification import MulticlassCalibrationError

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import real_amplitudes
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import SPSAEstimatorGradient
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient

SEED = 17
CURRENT_SEED = SEED
N_QUBITS = 8
COMPONENTS = [32, 16, 8, 4]

os.makedirs('../compressedFeatures', exist_ok=True)
os.makedirs('../artifacts', exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Seed impostato a {SEED}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")

Seed impostato a 17
CUDA disponibile: True


# 1. Config File

In [4]:
CFG = {
    "N_QUBITS": 8,
    "COMPONENTS": [32, 16, 8, 4],
    
    "ANSATZ": {
        "reps": 2,
        # 'linear', 'circular' (ring), 'ladder', 'graph_ladder'
    },
    
    "READOUT": {
        "type": "x_y_z_pair_nn",
    },
    
    "VQC": {
        "target_classes": 4,
        "use_gpu": True,
        "spsa_epsilon": 1e-6,
        "spsa_batch_size": 2,
        "estimator_precision": 0.0,
        "seed": 17,
    },
    
    "TRAINING": {
        "epochs": 200,
        "batch_size": 128,
        "learning_rate": 0.01,
        "weight_decay": 1e-4,
        "patience": 20,
    },
    
    "SEED": 17,
    "VERBOSE": True,
}

TOPOLOGIES = ["circular", "ladder", "graph_ladder", "full"]
print("Configurazione caricata")
print(f"Qubit: {CFG['N_QUBITS']}")
print(f"Dimensioni PCA: {CFG['COMPONENTS']}")

Configurazione caricata
Qubit: 8
Dimensioni PCA: [32, 16, 8, 4]


# 2. Utility
- zero padding

In [5]:
def apply_padding(X, d, n_qubits):
    """Aggiunge zero padding se d non è multiplo di n_qubits"""
    n_blocks = math.ceil(d / n_qubits)
    padded_size = n_blocks * n_qubits
    if padded_size == d:
        return X
    pad = torch.zeros(X.shape[0], padded_size - d)
    return torch.cat([X, pad], dim=1)

In [6]:
def angular_scaling(X_train, X_val, X_test, range_min=-np.pi/2, range_max=np.pi/2):
    """
    Scala le feature nel range [range_min, range_max].
    Default: [-π/2, π/2]
    
    Args:
        X_train: training set
        X_val: validation set
        X_test: test set
        range_min: minimo del range di destinazione (default: -π/2)
        range_max: massimo del range di destinazione (default: π/2)
    """
    scaler = MinMaxScaler(feature_range=(range_min, range_max))
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Clipping per sicurezza (opzionale, con il nuovo range)
    X_val_scaled = np.clip(X_val_scaled, range_min, range_max)
    X_test_scaled = np.clip(X_test_scaled, range_min, range_max)
    
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler


# Processa ogni dimensione
for d in COMPONENTS:
    path = f'../compressedFeatures/pca_d{d}_seed_{CURRENT_SEED}.pt'
    ckpt = torch.load(path, weights_only=False)
    
    X_train = ckpt['train'].numpy()
    X_val = ckpt['val'].numpy()
    X_test = ckpt['test'].numpy()
    
    # Angular scaling con range [-π/2, π/2]
    X_train_ang, X_val_ang, X_test_ang, scaler = angular_scaling(
        X_train, X_val, X_test,
        range_min=-np.pi/2,
        range_max=np.pi/2
    )
    
    print(f'd={d:2d}')
    print(f'  range train: [{X_train_ang.min():.4f}, {X_train_ang.max():.4f}]')
    print(f'  range val:   [{X_val_ang.min():.4f}, {X_val_ang.max():.4f}]')
    print(f'  range test:  [{X_test_ang.min():.4f}, {X_test_ang.max():.4f}]\n')
    
    # Salva con nuovo nome per non sovrascrivere
    torch.save({
        'train_ang': torch.from_numpy(X_train_ang).float(),
        'val_ang': torch.from_numpy(X_val_ang).float(),
        'test_ang': torch.from_numpy(X_test_ang).float(),
        'y_train': ckpt['y_train'],
        'y_val': ckpt['y_val'],
        'y_test': ckpt['y_test'],
        'd': d,
        'seed': CURRENT_SEED,
        'angular_scaling': 'minmax_[-pi/2,pi/2]'  # aggiorna la descrizione
    }, f'../compressedFeatures/angular_d{d}_seed_{CURRENT_SEED}_minus_pi_half.pt')

d=32
  range train: [-1.5708, 1.5708]
  range val:   [-1.5708, 1.5708]
  range test:  [-1.5708, 1.5708]

d=16
  range train: [-1.5708, 1.5708]
  range val:   [-1.5708, 1.5708]
  range test:  [-1.5708, 1.5708]

d= 8
  range train: [-1.5708, 1.5708]
  range val:   [-1.5708, 1.5708]
  range test:  [-1.5708, 1.4156]

d= 4
  range train: [-1.5708, 1.5708]
  range val:   [-1.5708, 1.5708]
  range test:  [-1.5511, 0.9392]



In [7]:
def check_clipping_impact(X_train, X_val, X_test, range_min=-np.pi/2, range_max=np.pi/2, verbose=True):
    """
    Quantifica l'impatto del clipping PRIMA di applicarlo: per ciascun set,
    calcola quanti valori (e quali feature) cadrebbero fuori dal range
    [range_min, range_max] dopo il MinMaxScaler fittato sul training.

    Utile per capire se il clipping su X_val/X_test è quasi un no-op
    (poche % fuori range) o se interviene pesantemente (molte % fuori range,
    quindi molto gradiente "spento" nei punti clippati durante l'inferenza/
    valutazione, e potenziale fonte del peggioramento osservato).

    Args:
        X_train, X_val, X_test: array GIA' scalati con il MinMaxScaler
                                 (es. X_train_ang, X_val_ang, X_test_ang
                                 prima del np.clip su val/test)
        range_min, range_max: bound del range target
        verbose: se True, stampa un report leggibile

    Returns:
        dict con statistiche per train/val/test:
            {
              'train': {'pct_out': float, 'n_out': int, 'n_total': int,
                        'pct_out_per_feature': np.ndarray, 'max_overshoot': float},
              'val':   {...},
              'test':  {...}
            }
        max_overshoot = quanto il valore più estremo supera il bordo
        (es. range_max=1.57, valore=2.1 -> overshoot=0.53)
    """
    results = {}
    sets = {'train': X_train, 'val': X_val, 'test': X_test}

    for name, X in sets.items():
        out_mask = (X < range_min) | (X > range_max)
        n_out = int(out_mask.sum())
        n_total = X.size
        pct_out = 100.0 * n_out / n_total

        # overshoot massimo (quanto sfora il bordo, in valore assoluto)
        overshoot_low = np.maximum(range_min - X, 0)
        overshoot_high = np.maximum(X - range_max, 0)
        max_overshoot = float(np.maximum(overshoot_low, overshoot_high).max())

        # quota di valori fuori range, per singola feature/colonna
        pct_out_per_feature = 100.0 * out_mask.mean(axis=0)

        results[name] = {
            'pct_out': pct_out,
            'n_out': n_out,
            'n_total': n_total,
            'pct_out_per_feature': pct_out_per_feature,
            'max_overshoot': max_overshoot,
        }

        if verbose:
            print(f"[{name}] valori fuori range [{range_min:.4f}, {range_max:.4f}]: "
                  f"{n_out}/{n_total} ({pct_out:.3f}%) | overshoot max: {max_overshoot:.4f}")
            top_feats = np.argsort(pct_out_per_feature)[::-1][:5]
            problematic = [(int(f), float(pct_out_per_feature[f])) for f in top_feats if pct_out_per_feature[f] > 0]
            if problematic:
                print(f"        feature più colpite (idx, %): {problematic}")

    return results

In [8]:
stats = check_clipping_impact(X_train_ang, X_val_ang, X_test_ang,
                                range_min=-np.pi/2, range_max=np.pi/2)

[train] valori fuori range [-1.5708, 1.5708]: 0/16000 (0.000%) | overshoot max: 0.0000
[val] valori fuori range [-1.5708, 1.5708]: 0/4000 (0.000%) | overshoot max: 0.0000
[test] valori fuori range [-1.5708, 1.5708]: 0/4000 (0.000%) | overshoot max: 0.0000


# 3. Costruzione del circuito 

In [5]:
def build_encoding_layer(n_qubits, block_idx):
    params = ParameterVector(f'x_{block_idx}', n_qubits)
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(params[i], i)
    return qc, params

def build_encoding_circuit(d, n_qubits):
    n_blocks = math.ceil(d / n_qubits)
    qc = QuantumCircuit(n_qubits)
    all_input_params = []
    for block in range(n_blocks):
        enc_layer, enc_params = build_encoding_layer(n_qubits, block)
        qc.compose(enc_layer, inplace=True)
        all_input_params.extend(enc_params)
    return qc, all_input_params, n_blocks

def build_ansatz_layer(n_qubits, reps, entanglement, block_idx=0, shared=True):
    """
    Costruisce un ansatz RealAmplitudes.
    
    Args:
        n_qubits: numero di qubit
        reps: numero di ripetizioni
        entanglement: 'linear', 'circular', 'ladder', 'graph_ladder', 'full', o lista di coppie
        block_idx: indice del blocco (per parametri condivisi)
        shared: True per condividere i parametri tra blocchi
    """
    prefix = 'θ' if shared else f'θ_{block_idx}'
    
    # Gestione per ladder e graph_ladder
    if entanglement in ["ladder", "graph_ladder"]:
        if n_qubits % 2 != 0:
            raise ValueError(f"Ladder richiede n_qubits pari, ma n={n_qubits}")
        k = n_qubits // 2
        pattern = []
        
        # Corda superiore (qubit 0..k-1)
        for i in range(k - 1):
            pattern.append([i, i+1])
        
        # Corda inferiore (qubit k..n_qubits-1)
        for i in range(k, n_qubits - 1):
            pattern.append([i, i+1])
        
        # Pioli verticali (sempre presenti)
        for i in range(k):
            pattern.append([i, i + k])
        
        # CONNESSIONI DIAGONALI (solo per graph_ladder)
        if entanglement == "graph_ladder":
            # Diagonali destra (dall'alto verso il basso a destra)
            for i in range(k - 1):
                pattern.append([i, i + k + 1])
            # Diagonali sinistra (dall'alto verso il basso a sinistra)
            for i in range(1, k):
                pattern.append([i, i + k - 1])
        
        ansatz = real_amplitudes(
            num_qubits=n_qubits,
            reps=reps,
            entanglement=pattern,
            parameter_prefix=prefix
        )
    
    elif entanglement == "full":
        # Full entanglement: tutte le coppie possibili
        ansatz = real_amplitudes(
            num_qubits=n_qubits,
            reps=reps,
            entanglement="full",
            parameter_prefix=prefix
        )
    
    elif entanglement == "circular":
        ansatz = real_amplitudes(
            num_qubits=n_qubits,
            reps=reps,
            entanglement="circular",
            parameter_prefix=prefix
        )
    
    else:  # linear o altri
        ansatz = real_amplitudes(
            num_qubits=n_qubits,
            reps=reps,
            entanglement=entanglement,
            parameter_prefix=prefix
        )
    
    return ansatz


def build_vqc_circuit(d, n_qubits, topology, reps=1, shared_params=True):
    """Aggiunto parametro topology"""
    n_blocks = math.ceil(d / n_qubits)
    qc = QuantumCircuit(n_qubits)
    all_input_params = []
    
    ansatz_template = build_ansatz_layer(n_qubits, reps, topology, block_idx=0, shared=True)
    
    for block in range(n_blocks):
        enc_layer, enc_params = build_encoding_layer(n_qubits, block)
        qc.compose(enc_layer, inplace=True)
        all_input_params.extend(enc_params)
        
        if shared_params:
            qc.compose(ansatz_template, inplace=True)
        else:
            ansatz_block = build_ansatz_layer(n_qubits, reps, topology, block, shared=False)
            qc.compose(ansatz_block, inplace=True)
    
    all_variational_params = list(ansatz_template.parameters)
    return qc, all_input_params, all_variational_params, n_blocks


# 4. Costruzione del ReadOut

In [6]:
def single_pauli_observables(n_qubits, op):
    observables = []
    for qubit in range(n_qubits):
        pauli = ["I"] * n_qubits
        pauli[n_qubits - 1 - qubit] = op
        observables.append(SparsePauliOp.from_list([("".join(pauli), 1.0)]))
    return observables

def pair_pauli_observables(n_qubits, op, pairs):
    observables = []
    for (i, j) in pairs:
        pauli = ["I"] * n_qubits
        pauli[n_qubits - 1 - i] = op
        pauli[n_qubits - 1 - j] = op
        observables.append(SparsePauliOp.from_list([("".join(pauli), 1.0)]))
    return observables

def get_nn_pairs(n_qubits, topology):
    pairs = []
    if topology == "linear":
        for i in range(n_qubits - 1):
            pairs.append((i, i+1))
    elif topology == "ring" or topology == "circular":
        for i in range(n_qubits - 1):
            pairs.append((i, i+1))
        pairs.append((n_qubits - 1, 0))
    elif topology in ["ladder", "graph_ladder"]:
        if n_qubits % 2 != 0:
            raise ValueError(f"Ladder richiede n_qubits pari, ma n={n_qubits}")
        k = n_qubits // 2
        for i in range(k - 1):
            pairs.append((i, i+1))
        for i in range(k, n_qubits - 1):
            pairs.append((i, i+1))
        for i in range(k):
            pairs.append((i, i + k))
        if topology == "graph_ladder":
            for i in range(k - 1):
                pairs.append((i, i + k + 1))
            for i in range(1, k):
                pairs.append((i, i + k - 1))
    elif topology == "full":
        pairs = [(i, j) for i in range(n_qubits) for j in range(i+1, n_qubits)]
    return pairs

def build_readout_observables(n_qubits, readout="z", topology="linear"):
    singles_xyz = (single_pauli_observables(n_qubits, "X")
                 + single_pauli_observables(n_qubits, "Y")
                 + single_pauli_observables(n_qubits, "Z"))
    
    if readout == "z":
        return single_pauli_observables(n_qubits, "Z")
    elif readout == "x_z":
        return (single_pauli_observables(n_qubits, "X")
              + single_pauli_observables(n_qubits, "Z"))
    elif readout == "x_y_z":
        return singles_xyz
    elif readout == "x_y_z_pair_nn":
        nn_pairs = get_nn_pairs(n_qubits, topology)
        pair_obs = []
        for op in ["X", "Y", "Z"]:
            pair_obs += pair_pauli_observables(n_qubits, op, nn_pairs)
        return singles_xyz + pair_obs
    elif readout == "x_y_z_pair_all":
        all_pairs = [(i, j) for i in range(n_qubits) for j in range(i+1, n_qubits)]
        pair_obs = []
        for op in ["X", "Y", "Z"]:
            pair_obs += pair_pauli_observables(n_qubits, op, all_pairs)
        return singles_xyz + pair_obs
    else:
        raise ValueError(f"Readout non supportato: {readout}")

# ============================================
# TEST PER N_QUBITS = 8
# ============================================
N_QUBITS = 8
topologies = ["linear", "ring", "ladder", "graph_ladder", "full"]

print(f"\n{'='*60}")
print(f"CONFRONTO TOPOLOGIE per n_qubits = {N_QUBITS}")
print(f"{'='*60}\n")

for topology in topologies:
    if topology in ["ladder", "graph_ladder"] and N_QUBITS % 2 != 0:
        print(f"{topology}: skip (n_qubits deve essere pari)")
        continue

    print(f"\n📐 Topologia: {topology.upper()}")
    print(f"{'-'*40}")

    nn_pairs = get_nn_pairs(N_QUBITS, topology)
    print(f"  Coppie NN ({len(nn_pairs)}): {nn_pairs}")

    for readout in ["z", "x_z", "x_y_z", "x_y_z_pair_nn", "x_y_z_pair_all"]:
        obs = build_readout_observables(N_QUBITS, readout, topology)
        print(f"  readout={readout:20s} → {len(obs):3d} osservabili")


CONFRONTO TOPOLOGIE per n_qubits = 8


📐 Topologia: LINEAR
----------------------------------------
  Coppie NN (7): [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)]
  readout=z                    →   8 osservabili
  readout=x_z                  →  16 osservabili
  readout=x_y_z                →  24 osservabili
  readout=x_y_z_pair_nn        →  45 osservabili
  readout=x_y_z_pair_all       → 108 osservabili

📐 Topologia: RING
----------------------------------------
  Coppie NN (8): [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 0)]
  readout=z                    →   8 osservabili
  readout=x_z                  →  16 osservabili
  readout=x_y_z                →  24 osservabili
  readout=x_y_z_pair_nn        →  48 osservabili
  readout=x_y_z_pair_all       → 108 osservabili

📐 Topologia: LADDER
----------------------------------------
  Coppie NN (10): [(0, 1), (1, 2), (2, 3), (4, 5), (5, 6), (6, 7), (0, 4), (1, 5), (2, 6), (3, 7)]
  readout=z                    →

# 5. Classe VQC con SPSA

In [7]:
class _BatchedEstimatorPauliSPSAFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, model, x, weights):
        features = model._estimate_readout_features(
            x.detach().cpu().numpy().astype(np.float64),
            weights.detach().cpu().numpy().astype(np.float64),
        )
        ctx.model = model
        ctx.save_for_backward(x.detach(), weights.detach())
        return torch.tensor(features, dtype=x.dtype, device=x.device)

    @staticmethod
    def backward(ctx, grad_output):
        x, weights = ctx.saved_tensors
        model = ctx.model
        grad_weights = model._spsa_weight_gradient(
            x.detach().cpu().numpy().astype(np.float64),
            weights.detach().cpu().numpy().astype(np.float64),
            grad_output.detach().cpu().numpy().astype(np.float64),
        )
        return None, None, torch.tensor(grad_weights, dtype=weights.dtype, device=weights.device)


class VQC(nn.Module):
    def __init__(self, quantum_circuit, observables, input_params, weight_params,
                 readout_dim, target_classes=4, seed=SEED,
                 spsa_epsilon=0.1, spsa_batch_size=1,
                 estimator_precision=0.0, use_gpu=True):
        super().__init__()
        
        self.quantum_circuit = quantum_circuit
        self.input_params = input_params
        self.weight_params = weight_params
        self.readout_dim = readout_dim
        self.spsa_epsilon = float(spsa_epsilon)
        self.spsa_batch_size = int(spsa_batch_size)
        self.estimator_precision = float(estimator_precision)
        self._spsa_rng = np.random.default_rng(seed)
        
        self.estimator = AerEstimator()
        sim_opts = {"method": "statevector"}
        if use_gpu:
            try:
                sim_opts.update({"device": "GPU", "cuStateVec_enable": True})
                print("  GPU attiva: cuStateVec abilitato")
            except Exception as e:
                print(f"  GPU non disponibile ({e}), uso CPU")
        self.estimator.options.simulator = sim_opts
        
        np.random.seed(seed)
        init_w = np.random.uniform(-0.01, 0.01, len(list(weight_params))).astype(np.float32)
        self.q_weights = nn.Parameter(torch.tensor(init_w))
        
        self.head = nn.Linear(readout_dim, target_classes)
        
        self.observables = [[obs] for obs in observables]
        
        source_params = list(input_params) + list(weight_params)
        source_index = {p: i for i, p in enumerate(source_params)}
        self._parameter_order = list(quantum_circuit.parameters)
        try:
            self._parameter_source_indices = np.array(
                [source_index[p] for p in self._parameter_order], dtype=np.int64
            )
        except KeyError as e:
            raise RuntimeError(f"Parametro non tracciato nel circuito: {e}")
        
        print(f'  q_weights : {self.q_weights.shape}')
        print(f'  head      : Linear({readout_dim}, {target_classes})')
    
    def _ordered_parameter_values(self, input_values, weights):
        if input_values.ndim == 1:
            input_values = input_values.reshape(1, -1)
        if weights.ndim == 1:
            weight_values = np.broadcast_to(weights, (input_values.shape[0], weights.shape[0]))
        else:
            weight_values = weights
        source_values = np.concatenate([input_values, weight_values], axis=1)
        return source_values[:, self._parameter_source_indices]
    
    def _estimate_readout_features(self, input_values, weights):
        parameter_values = self._ordered_parameter_values(input_values, weights)
        pub = (self.quantum_circuit, self.observables, parameter_values)
        result = self.estimator.run([pub], precision=self.estimator_precision).result()
        evs = np.asarray(result[0].data.evs, dtype=np.float64)
        expected_size = self.readout_dim * input_values.shape[0]
        if evs.size != expected_size:
            raise RuntimeError(f"evs size={evs.size}, atteso {expected_size}")
        return evs.reshape(self.readout_dim, input_values.shape[0]).T
    
    def _spsa_weight_gradient(self, input_values, weights, upstream_gradient):
        deltas = self._spsa_rng.choice(
            np.array([-1.0, 1.0], dtype=np.float64),
            size=(self.spsa_batch_size, weights.shape[0])
        )
        perturbed = []
        for delta in deltas:
            perturbed.append(weights + self.spsa_epsilon * delta)
            perturbed.append(weights - self.spsa_epsilon * delta)
        
        perturbed_weights = np.repeat(np.stack(perturbed), input_values.shape[0], axis=0)
        repeated_inputs = np.tile(input_values, (len(perturbed), 1))
        features = self._estimate_readout_features(repeated_inputs, perturbed_weights)
        features = features.reshape(len(perturbed), input_values.shape[0], -1)
        
        grad = np.zeros_like(weights, dtype=np.float64)
        for idx, delta in enumerate(deltas):
            plus = features[2 * idx]
            minus = features[2 * idx + 1]
            directional = np.sum(upstream_gradient * (plus - minus)) / (2.0 * self.spsa_epsilon)
            grad += directional * delta
        return grad / float(self.spsa_batch_size)
    
    def forward(self, x):
        q_out = _BatchedEstimatorPauliSPSAFunction.apply(self, x, self.q_weights)
        return self.head(q_out)


def build_model(d, n_qubits, topology, reps=1, readout="z", seed=SEED,
                spsa_epsilon=0.1, spsa_batch_size=1, use_gpu=True):
    qc, input_params, var_params, _ = build_vqc_circuit(d, n_qubits, topology, reps)
    observables = build_readout_observables(n_qubits, readout, topology)
    print(f'd={d:2d} | topology={topology} | readout={readout} | var_params={len(var_params)} | readout_dim={len(observables)}')
    model = VQC(
        quantum_circuit=qc, observables=observables,
        input_params=input_params, weight_params=var_params,
        readout_dim=len(observables), target_classes=4, seed=seed,
        spsa_epsilon=spsa_epsilon, spsa_batch_size=spsa_batch_size,
        use_gpu=use_gpu
    )
    return model

In [8]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit.library import RealAmplitudes

def draw_circuit_large(qc, title, filename, figsize=(25, 10), scale=0.35, fold=150):
    """
    Disegna un circuito quantistico con figura grande e salva come PNG.
    
    Args:
        qc: QuantumCircuit da disegnare
        title: titolo del circuito
        filename: nome del file di output
        figsize: dimensione della figura (larghezza, altezza)
        scale: scala di disegno (più piccolo = più compatto)
        fold: numero di caratteri prima di andare a capo nel testo
    """
    fig, ax = plt.subplots(figsize=figsize)
    qc.draw('mpl', style='clifford', scale=scale, fold=fold, ax=ax)
    ax.set_title(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)  # Chiude la figura per liberare memoria
    print(f"✅ Salvato: {filename}")

# ============================================
# PARAMETRI
# ============================================
n_qubits = 8
reps = 2
d = 32  # dummy, solo per la struttura

# ============================================
# 1. CIRCUITO AD ANELLO (RING)
# ============================================
print("Costruzione circuito ad anello (ring)...")
qc_ring, _, _, _ = build_vqc_circuit(
    d=d, 
    n_qubits=n_qubits, 
    topology='circular',  # ring in Qiskit
    reps=reps
)

print("\n" + "=" * 70)
print("CIRCUITO AD ANELLO (RING) - 8 qubit, reps=2")
print("=" * 70)

# Versione testuale (per ispezione rapida)
print(qc_ring.draw('text', fold=150))

# Versione grafica (salva come immagine)
draw_circuit_large(
    qc_ring, 
    title=f"Circuito ad Anello (Ring) - {n_qubits} qubit, reps={reps}",
    filename="circuito_ring_8qubit.png",
    figsize=(25, 10),
    scale=0.35,
    fold=150
)

# ============================================
# 2. CIRCUITO A SCALA (LADDER)
# ============================================
print("\n" + "=" * 70)
print("CIRCUITO A SCALA (LADDER) - 8 qubit, reps=2")
print("=" * 70)

qc_ladder, _, _, _ = build_vqc_circuit(
    d=d, 
    n_qubits=n_qubits, 
    topology='ladder',
    reps=reps
)

# Versione testuale
print(qc_ladder.draw('text', fold=150))

# Versione grafica
draw_circuit_large(
    qc_ladder, 
    title=f"Circuito a Scala (Ladder) - {n_qubits} qubit, reps={reps}",
    filename="circuito_ladder_8qubit.png",
    figsize=(25, 10),
    scale=0.35,
    fold=150
)

# ============================================
# 3. RIEPILOGO FILE SALVATI
# ============================================
print("\n" + "=" * 70)
print("FILE SALVATI:")
print("=" * 70)
print("  📁 circuito_ring_8qubit.png   - Circuito ad anello (ring)")
print("  📁 circuito_ladder_8qubit.png - Circuito a scala (ladder)")

Costruzione circuito ad anello (ring)...

CIRCUITO AD ANELLO (RING) - 8 qubit, reps=2
     ┌────────────┐┌──────────┐┌───┐     ┌──────────┐                                                                             ┌───┐     »
q_0: ┤ Ry(x_0[0]) ├┤ Ry(θ[0]) ├┤ X ├──■──┤ Ry(θ[8]) ├─────────────────────────────────────────────────────────────────────────────┤ X ├──■──»
     ├────────────┤├──────────┤└─┬─┘┌─┴─┐└──────────┘┌──────────┐                                                                 └─┬─┘┌─┴─┐»
q_1: ┤ Ry(x_0[1]) ├┤ Ry(θ[1]) ├──┼──┤ X ├─────■──────┤ Ry(θ[9]) ├───────────────────────────────────────────────────────────────────┼──┤ X ├»
     ├────────────┤├──────────┤  │  └───┘   ┌─┴─┐    └──────────┘┌───────────┐                                                      │  └───┘»
q_2: ┤ Ry(x_0[2]) ├┤ Ry(θ[2]) ├──┼──────────┤ X ├─────────■──────┤ Ry(θ[10]) ├──────────────────────────────────────────────────────┼───────»
     ├────────────┤├──────────┤  │          └───┘       ┌─┴─┐ 

# 6. Training e valutazione

In [9]:
def train_vqc(d, n_qubits, topology, n_epochs=50, batch_size=128, patience=15,
              seed=SEED, spsa_epsilon=0.1, spsa_batch_size=1,
              readout="z", reps=1, learning_rate=0.01, weight_decay=1e-4,
              log_every=10, n_classes=4):
    """
    🔴 MODIFICA: il checkpoint del modello ora si basa sul macro F1 score
    calcolato sul VALIDATION set (non sul test, che deve restare intoccato
    fino alla valutazione finale per evitare data leakage).
    Ogni `log_every` epoche viene stampato un riepilogo completo delle
    metriche (f1, auroc, balanced accuracy, ece) su validation.
    """

    # Carica i dati con range [-pi/2, pi/2]
    ckpt = torch.load(f'../compressedFeatures/angular_d{d}_seed_{seed}_minus_pi_half.pt', weights_only=False)
    X_train = apply_padding(ckpt['train_ang'], d, n_qubits)
    X_val = apply_padding(ckpt['val_ang'], d, n_qubits)
    X_test = apply_padding(ckpt['test_ang'], d, n_qubits)
    y_train = ckpt['y_train'].squeeze().long()
    y_val = ckpt['y_val'].squeeze().long()
    y_test = ckpt['y_test'].squeeze().long()

    train_loader = DataLoader(TensorDataset(X_train, y_train),
                              batch_size=batch_size, shuffle=True)

    model = build_model(d, n_qubits, topology, reps=reps, readout=readout, seed=seed,
                        spsa_epsilon=spsa_epsilon, spsa_batch_size=spsa_batch_size)

    optimizer = torch.optim.Adam([
        {'params': [model.q_weights],       'lr': learning_rate},
        {'params': model.head.parameters(), 'lr': learning_rate}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-4
    )
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    # 🔴 MODIFICA: il "best" ora è il massimo macro_f1 su validation
    # (prima era il minimo val_loss)
    best_val_f1 = -float('inf')
    best_val_metrics = None
    patience_counter = 0
    best_weights = None
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': [],
        'val_auroc': [],
        'val_bacc': [],
        'val_ece': []
    }

    print(f'd={d} | topology={topology} | epochs={n_epochs}')
    print('-' * 50)

    for epoch in range(n_epochs):
        model.train()
        epoch_losses = []

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        train_loss = np.mean(epoch_losses)

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()

        # 🔴 MODIFICA: calcolo di tutte le metriche di interesse su validation
        # (riusa evaluate_vqc, definita più sotto), una volta per epoca
        val_metrics, _, _ = evaluate_vqc(model, X_val, y_val, n_classes=n_classes)
        val_f1 = val_metrics['macro_f1']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_auroc'].append(val_metrics['macro_auroc'])
        history['val_bacc'].append(val_metrics['bal_acc'])
        history['val_ece'].append(val_metrics['ece'])

        # 🔴 MODIFICA: log periodico con TUTTE le metriche interessanti
        if CFG["VERBOSE"] and (epoch % log_every == 0 or epoch == n_epochs - 1):
            print(f'epoch {epoch+1:3d}/{n_epochs} | '
                  f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
                  f'val_f1={val_f1:.4f} | val_auroc={val_metrics["macro_auroc"]:.4f} | '
                  f'val_bacc={val_metrics["bal_acc"]:.4f} | val_ece={val_metrics["ece"]:.4f}')

        # 🔴 MODIFICA: checkpoint sul macro F1 di validation (più alto è meglio)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_metrics = val_metrics
            patience_counter = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping a epoch {epoch+1}')
                break

        scheduler.step()

    model.load_state_dict(best_weights)
    print(f'Miglior val_f1: {best_val_f1:.4f} | '
          f'val_auroc={best_val_metrics["macro_auroc"]:.4f} | '
          f'val_bacc={best_val_metrics["bal_acc"]:.4f} | '
          f'val_ece={best_val_metrics["ece"]:.4f}')

    return model, history, X_val, y_val, X_test, y_test


def evaluate_vqc(model, X, y, n_classes=4):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.softmax(logits, dim=1).numpy()
        preds = probs.argmax(axis=1)
    y_np = y.numpy()

    auroc = roc_auc_score(y_np, probs, multi_class='ovr', average='macro')
    f1 = f1_score(y_np, preds, average='macro', zero_division=0)
    bacc = balanced_accuracy_score(y_np, preds)

    ece_metric = MulticlassCalibrationError(num_classes=n_classes, n_bins=15, norm='l1')
    ece = ece_metric(torch.tensor(probs, dtype=torch.float32),
                     torch.tensor(y_np, dtype=torch.long)).item()

    return {
        'macro_auroc': round(auroc, 4),
        'macro_f1': round(f1, 4),
        'bal_acc': round(bacc, 4),
        'ece': round(ece, 4)
    }, preds, probs


def save_results(model, d, n_qubits, topology, readout, reps, spsa_batch_size,
                  seed, n_epochs_requested, history, val_metrics, test_metrics,
                  elapsed_min):
    """
    Salva i risultati di un esperimento in file separati (json + npz + pt),
    in una sottocartella dedicata alla topologia.

    🔴 FIX rispetto alla versione precedente:
    - readout/reps/spsa_batch_size sono passati come parametri espliciti,
      non più ricavati con .split('_') sul tag (che falliva con readout
      contenenti underscore, es. 'x_y_z_pair_nn').
    - i pesi sono salvati con estensione .pt (erano salvati come .json
      anche se torch.save produce un binario).
    - save_dir include la topologia, così esperimenti di topologie diverse
      non si sovrascrivono e il controllo "già presente" nello script
      principale punta alla cartella giusta.
    - gli angoli (X_val/X_test) NON vengono duplicati per ogni esperimento:
      dipendono solo da d e seed (non da topology/readout/reps), quindi nel
      json salviamo solo il riferimento al file da cui sono stati caricati.
      Il nome del file di output, però, contiene topology, d e seed, così
      ogni esperimento resta univocamente identificabile.
    """
    tag = f"d{d}_{readout}_reps{reps}"

    save_dir = f'../artifacts/c3_pi_minus_half/{topology}'
    os.makedirs(save_dir, exist_ok=True)

    base_name = f'vqc_{tag}_topo_{topology}_seed_{seed}'

    n_epochs_run = len(history['train_loss'])  # epoche effettive (early stopping incluso)
    angular_data_path = f'../compressedFeatures/angular_d{d}_seed_{seed}_minus_pi_half.pt'

    artifact = {
        # --- identificativi che differenziano un esperimento dall'altro ---
        'd': d,
        'topology': topology,
        'readout': readout,
        'reps': reps,
        'spsa_batch_size': spsa_batch_size,
        'seed': seed,
        'n_qubits': n_qubits,
        'model': 'VQC',
        'ansatz': 'realAmpl',
        'encoding_range': '[-pi/2, pi/2]',
        # --- riferimento agli angoli (NON duplicati: dipendono solo da d, seed) ---
        'angular_data_path': angular_data_path,
        # --- training ---
        'n_epochs_requested': n_epochs_requested,
        'n_epochs_run': n_epochs_run,
        'early_stopped': n_epochs_run < n_epochs_requested,
        'elapsed_min': round(elapsed_min, 2),
        # --- metriche ---
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        # --- riferimenti agli altri file dello stesso esperimento ---
        'weights_file': f'{base_name}.pt',
        'history_file': f'{base_name}_history.npz',
    }

    # 1. metadati + config + metriche
    with open(f'{save_dir}/{base_name}.json', 'w') as f:
        json.dump(artifact, f, indent=2)

    # 2. curve di training/validation (tutte le metriche per epoca)
    np.savez(f'{save_dir}/{base_name}_history.npz',
             train_loss=np.array(history['train_loss']),
             val_loss=np.array(history['val_loss']),
             val_f1=np.array(history.get('val_f1', [])),
             val_auroc=np.array(history.get('val_auroc', [])),
             val_bacc=np.array(history.get('val_bacc', [])),
             val_ece=np.array(history.get('val_ece', [])))

    # 3. pesi del modello (🔴 FIX: estensione .pt, non .json)
    torch.save(model.state_dict(), f'{save_dir}/{base_name}.pt')

    print(f'✅ Salvato: {save_dir}/{base_name}.json')
    print(f'   topology={topology} | d={d} | readout={readout} | reps={reps} | '
          f'test_f1={test_metrics["macro_f1"]:.4f}')

# 7. Esperimenti  

## 7.1 Sanity Check e Benchmark

In [10]:
def sanity_check(d=4, topology="ring", readout="x_z", reps=1, n_epochs=3):
    """
    Sanity check rapido per verificare che tutto funzioni.
    - d=4: dimensione minima
    - topology: 'linear', 'ring', 'ladder', 'graph-ladder', 'full'
    - readout: 'z', 'x_z', 'x_y_z', 'x_y_z_pair_nn'
    - reps: 1 (shallow)
    - n_epochs: solo 3 epoche
    """
    print("=" * 60)
    print("SANITY CHECK")
    print("=" * 60)
    print(f"Configurazione:")
    print(f"  d={d}")
    print(f"  topology={topology}")
    print(f"  readout={readout}")
    print(f"  reps={reps}")
    print(f"  n_epochs={n_epochs}")
    print("-" * 60)
    
    try:
        # 1. Verifica caricamento dati
        print("\n[1/5] Caricamento dati...")
        ckpt_path = f'../compressedFeatures/angular_d{d}_seed_{SEED}_minus_pi_half.pt'
        if not os.path.exists(ckpt_path):
            print(f"  ❌ File non trovato: {ckpt_path}")
            return False
        
        ckpt = torch.load(ckpt_path, weights_only=False)
        X_train = apply_padding(ckpt['train_ang'], d, N_QUBITS)
        y_train = ckpt['y_train'].squeeze().long()
        print(f"  ✅ X_train shape: {X_train.shape}")
        print(f"  ✅ y_train shape: {y_train.shape}")
        
        # 2. Verifica costruzione circuito
        print("\n[2/5] Costruzione circuito...")
        qc, input_params, var_params, n_blocks = build_vqc_circuit(d, N_QUBITS, topology, reps)
        print(f"  ✅ Circuito: {qc.num_qubits} qubit, {qc.size()} porte")
        print(f"  ✅ Input params: {len(input_params)}")
        print(f"  ✅ Var params: {len(var_params)}")
        print(f"  ✅ Blocchi: {n_blocks}")
        
        # 3. Verifica osservabili
        print("\n[3/5] Costruzione osservabili...")
        observables = build_readout_observables(N_QUBITS, readout, topology)
        print(f"  ✅ {len(observables)} osservabili")
        
        # 4. Verifica modello
        print("\n[4/5] Creazione modello...")
        model = build_model(d, N_QUBITS, topology, reps=reps, readout=readout, seed=SEED,
                            spsa_epsilon=0.1, spsa_batch_size=1, use_gpu=True)
        print(f"  ✅ Modello creato")
        print(f"  ✅ q_weights: {model.q_weights.shape}")
        print(f"  ✅ head: {model.head}")
        
        # 5. Verifica training loop (3 epoche)
        print("\n[5/5] Test training loop (3 epoche)...")
        train_loader = DataLoader(TensorDataset(X_train[:64], y_train[:64]), batch_size=16, shuffle=True)
        
        optimizer = torch.optim.Adam([
            {'params': [model.q_weights], 'lr': 0.01},
            {'params': model.head.parameters(), 'lr': 0.01}
        ])
        loss_fn = nn.CrossEntropyLoss()
        
        t_start = time.perf_counter()
        for epoch in range(3):
            epoch_losses = []
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                logits = model(X_batch)
                loss = loss_fn(logits, y_batch)
                loss.backward()
                optimizer.step()
                epoch_losses.append(loss.item())
            print(f"  Epoch {epoch+1}: loss={np.mean(epoch_losses):.4f}")
        
        elapsed = time.perf_counter() - t_start
        print(f"\n✅ SANITY CHECK SUPERATO! (tempo: {elapsed:.1f} secondi)")
        return True
        
    except Exception as e:
        print(f"\n❌ SANITY CHECK FALLITO: {e}")
        import traceback
        traceback.print_exc()
        return False

# Esegui sanity check
sanity_check(d=4, topology="linear", readout="x_z", reps=1)

SANITY CHECK
Configurazione:
  d=4
  topology=linear
  readout=x_z
  reps=1
  n_epochs=3
------------------------------------------------------------

[1/5] Caricamento dati...
  ✅ X_train shape: torch.Size([4000, 8])
  ✅ y_train shape: torch.Size([4000])

[2/5] Costruzione circuito...
  ✅ Circuito: 8 qubit, 31 porte
  ✅ Input params: 8
  ✅ Var params: 16
  ✅ Blocchi: 1

[3/5] Costruzione osservabili...
  ✅ 16 osservabili

[4/5] Creazione modello...
d= 4 | topology=linear | readout=x_z | var_params=16 | readout_dim=16
  GPU attiva: cuStateVec abilitato
  q_weights : torch.Size([16])
  head      : Linear(16, 4)
  ✅ Modello creato
  ✅ q_weights: torch.Size([16])
  ✅ head: Linear(in_features=16, out_features=4, bias=True)

[5/5] Test training loop (3 epoche)...
  Epoch 1: loss=1.3760
  Epoch 2: loss=1.3591
  Epoch 3: loss=1.3478

✅ SANITY CHECK SUPERATO! (tempo: 0.3 secondi)


True

In [11]:
def benchmark_topology(d=16, topology="linear", readout="x_y_z_pair_nn", reps=2, n_batches=5):
    """
    Benchmark per stimare il tempo per epoca e il tempo totale stimato.
    """
    print("=" * 60)
    print("BENCHMARK")
    print("=" * 60)
    print(f"Configurazione benchmark:")
    print(f"  d={d}")
    print(f"  topology={topology}")
    print(f"  readout={readout}")
    print(f"  reps={reps}")
    print(f"  n_batches={n_batches} (solo per stima)")
    print("-" * 60)
    
    # Carica dati
    ckpt = torch.load(f'../compressedFeatures/angular_d{d}_seed_{SEED}_minus_pi_half.pt', weights_only=False)
    X_train = apply_padding(ckpt['train_ang'], d, N_QUBITS)
    y_train = ckpt['y_train'].squeeze().long()
    
    batch_size = 128
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    
    # Crea modello
    model = build_model(d, N_QUBITS, topology, reps=reps, readout=readout, seed=SEED,
                        spsa_epsilon=0.1, spsa_batch_size=2, use_gpu=True)
    
    optimizer = torch.optim.Adam([
        {'params': [model.q_weights], 'lr': 0.01},
        {'params': model.head.parameters(), 'lr': 0.01}
    ])
    loss_fn = nn.CrossEntropyLoss()
    
    # Benchmark su n_batches
    print(f"\nEsecuzione benchmark su {n_batches} batch...")
    batch_times = []
    
    for i, (X_batch, y_batch) in enumerate(train_loader):
        if i >= n_batches:
            break
        
        t0 = time.perf_counter()
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        batch_times.append(time.perf_counter() - t0)
        
        print(f"  Batch {i+1}: {batch_times[-1]:.2f}s")
    
    avg_batch_time = np.mean(batch_times)
    n_batches_per_epoch = len(train_loader)
    est_epoch_time = avg_batch_time * n_batches_per_epoch
    est_total_time = est_epoch_time * CFG["TRAINING"]["epochs"]
    
    print("\n" + "-" * 60)
    print("STIME:")
    print(f"  Tempo medio per batch: {avg_batch_time:.2f}s")
    print(f"  Batch per epoca: {n_batches_per_epoch}")
    print(f"  Tempo stimato per epoca: {est_epoch_time:.1f}s ({est_epoch_time/60:.2f} min)")
    print(f"  Tempo stimato totale ({CFG['TRAINING']['epochs']} epoche): {est_total_time/60:.1f} min")
    print("=" * 60)
    
    return {
        'avg_batch_time': avg_batch_time,
        'est_epoch_time': est_epoch_time,
        'est_total_time': est_total_time / 60  # in minuti
    }

# Esegui benchmark (opzionale, decommentare per eseguire)
benchmark_topology(d=16, topology="linear", readout="x_y_z_pair_nn", reps=2)

BENCHMARK
Configurazione benchmark:
  d=16
  topology=linear
  readout=x_y_z_pair_nn
  reps=2
  n_batches=5 (solo per stima)
------------------------------------------------------------
d=16 | topology=linear | readout=x_y_z_pair_nn | var_params=24 | readout_dim=45
  GPU attiva: cuStateVec abilitato
  q_weights : torch.Size([24])
  head      : Linear(45, 4)

Esecuzione benchmark su 5 batch...
  Batch 1: 0.45s
  Batch 2: 0.41s
  Batch 3: 0.40s
  Batch 4: 0.42s
  Batch 5: 0.41s

------------------------------------------------------------
STIME:
  Tempo medio per batch: 0.42s
  Batch per epoca: 32
  Tempo stimato per epoca: 13.3s (0.22 min)
  Tempo stimato totale (200 epoche): 44.5 min


{'avg_batch_time': np.float64(0.4169618580000005),
 'est_epoch_time': np.float64(13.342779456000017),
 'est_total_time': np.float64(44.47593152000005)}

In [12]:
def test_all_topologies(d=4, readout="x_z", reps=1):
    """
    Test rapido di tutte le topologie per verificare che funzionino.
    """
    print("=" * 60)
    print("TEST TUTTE LE TOPOLOGIE")
    print("=" * 60)
    
    topologies = ["linear", "circular", "ladder", "graph_ladder", "full"]
    results = {}
    
    for topology in topologies:
        print(f"\n--- Testing topology: {topology.upper()} ---")
        
        try:
            # Costruisci circuito
            qc, input_params, var_params, n_blocks = build_vqc_circuit(d, N_QUBITS, topology, reps)
            print(f"  ✅ Circuito: {qc.num_qubits} qubit, {qc.size()} porte")
            
            # Costruisci osservabili
            observables = build_readout_observables(N_QUBITS, readout, topology)
            print(f"  ✅ Osservabili: {len(observables)}")
            
            # Conta i CNOT/CZ
            n_cnot = qc.count_ops().get('cx', 0)
            n_cz = qc.count_ops().get('cz', 0)
            print(f"  ✅ Porte entanglement: CNOT={n_cnot}, CZ={n_cz}")
            
            results[topology] = {
                'n_qubits': N_QUBITS,
                'n_observables': len(observables),
                'n_var_params': len(var_params),
                'n_blocks': n_blocks,
                'n_entanglement_gates': n_cnot + n_cz,
                'status': 'OK'
            }
            
        except Exception as e:
            print(f"  ❌ ERRORE: {e}")
            results[topology] = {'status': 'ERROR', 'error': str(e)}
    
    # Riepilogo
    print("\n" + "=" * 60)
    print("RIEPILOGO TEST TOPOLOGIE")
    print("=" * 60)
    for topology, res in results.items():
        if res['status'] == 'OK':
            print(f"{topology:10s} | observables={res['n_observables']:3d} | var_params={res['n_var_params']:2d} | entanglement_gates={res['n_entanglement_gates']}")
        else:
            print(f"{topology:10s} | FAILED: {res.get('error', 'unknown')}")
    
    return results

# Esegui test
print("x_z")
test_all_topologies(d=4, readout="x_z", reps=1)  
print("x_y_z")
test_all_topologies(d=4, readout="x_y_z", reps=1)  

x_z
TEST TUTTE LE TOPOLOGIE

--- Testing topology: LINEAR ---
  ✅ Circuito: 8 qubit, 31 porte
  ✅ Osservabili: 16
  ✅ Porte entanglement: CNOT=7, CZ=0

--- Testing topology: CIRCULAR ---
  ✅ Circuito: 8 qubit, 32 porte
  ✅ Osservabili: 16
  ✅ Porte entanglement: CNOT=8, CZ=0

--- Testing topology: LADDER ---
  ✅ Circuito: 8 qubit, 34 porte
  ✅ Osservabili: 16
  ✅ Porte entanglement: CNOT=10, CZ=0

--- Testing topology: GRAPH_LADDER ---
  ✅ Circuito: 8 qubit, 40 porte
  ✅ Osservabili: 16
  ✅ Porte entanglement: CNOT=16, CZ=0

--- Testing topology: FULL ---
  ✅ Circuito: 8 qubit, 52 porte
  ✅ Osservabili: 16
  ✅ Porte entanglement: CNOT=28, CZ=0

RIEPILOGO TEST TOPOLOGIE
linear     | observables= 16 | var_params=16 | entanglement_gates=7
circular   | observables= 16 | var_params=16 | entanglement_gates=8
ladder     | observables= 16 | var_params=16 | entanglement_gates=10
graph_ladder | observables= 16 | var_params=16 | entanglement_gates=16
full       | observables= 16 | var_params=16 |

{'linear': {'n_qubits': 8,
  'n_observables': 24,
  'n_var_params': 16,
  'n_blocks': 1,
  'n_entanglement_gates': 7,
  'status': 'OK'},
 'circular': {'n_qubits': 8,
  'n_observables': 24,
  'n_var_params': 16,
  'n_blocks': 1,
  'n_entanglement_gates': 8,
  'status': 'OK'},
 'ladder': {'n_qubits': 8,
  'n_observables': 24,
  'n_var_params': 16,
  'n_blocks': 1,
  'n_entanglement_gates': 10,
  'status': 'OK'},
 'graph_ladder': {'n_qubits': 8,
  'n_observables': 24,
  'n_var_params': 16,
  'n_blocks': 1,
  'n_entanglement_gates': 16,
  'status': 'OK'},
 'full': {'n_qubits': 8,
  'n_observables': 24,
  'n_var_params': 16,
  'n_blocks': 1,
  'n_entanglement_gates': 28,
  'status': 'OK'}}

## 7.2 Esperimenti

In [13]:
from time import perf_counter

'''
test che salva per verificare che il salvataggio sia corretto
'''

t_start = perf_counter()

model, history, X_val, y_val, X_test, y_test = train_vqc(
    d=4,                    
    n_qubits=N_QUBITS,
    topology="circular",    
    n_epochs=5,              
    batch_size=128,
    patience=3,
    seed=SEED,
    spsa_epsilon=CFG["VQC"]["spsa_epsilon"],
    spsa_batch_size=1,
    readout="x_z",
    reps=1,
    learning_rate=0.01,
    weight_decay=1e-4,
    log_every=1              
)

val_metrics, _, _ = evaluate_vqc(model, X_val, y_val)
test_metrics, _, _ = evaluate_vqc(model, X_test, y_test)
elapsed_min = (perf_counter() - t_start) / 60

print("\nVAL:", val_metrics)
print("TEST:", test_metrics)
print(f"Tempo: {elapsed_min:.2f} min")

save_results(
    model=model, d=4, n_qubits=N_QUBITS,
    topology="test_run",    
    readout="x_z", reps=1, spsa_batch_size=1,
    seed=SEED, n_epochs_requested=5,
    history=history, val_metrics=val_metrics, test_metrics=test_metrics,
    elapsed_min=elapsed_min
)


d= 4 | topology=circular | readout=x_z | var_params=16 | readout_dim=16
  GPU attiva: cuStateVec abilitato
  q_weights : torch.Size([16])
  head      : Linear(16, 4)
d=4 | topology=circular | epochs=5
--------------------------------------------------
epoch   1/5 | train_loss=1.3560 | val_loss=1.2969 | val_f1=0.3074 | val_auroc=0.7393 | val_bacc=0.3920 | val_ece=0.0762
epoch   2/5 | train_loss=1.2800 | val_loss=1.2540 | val_f1=0.4818 | val_auroc=0.7674 | val_bacc=0.4930 | val_ece=0.1478
epoch   3/5 | train_loss=1.2491 | val_loss=1.2301 | val_f1=0.4982 | val_auroc=0.7886 | val_bacc=0.5090 | val_ece=0.1401
epoch   4/5 | train_loss=1.2255 | val_loss=1.2178 | val_f1=0.5041 | val_auroc=0.7942 | val_bacc=0.5120 | val_ece=0.1363
epoch   5/5 | train_loss=1.2196 | val_loss=1.2139 | val_f1=0.5141 | val_auroc=0.7956 | val_bacc=0.5220 | val_ece=0.1428
Miglior val_f1: 0.5141 | val_auroc=0.7956 | val_bacc=0.5220 | val_ece=0.1428

VAL: {'macro_auroc': 0.7956, 'macro_f1': 0.5141, 'bal_acc': 0.522, 'ec

In [14]:
import time
import os
import json
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from time import perf_counter

# ============================================
# CONFIGURAZIONE
# ============================================
SEED = 17
N_QUBITS = 8
CFG = {
    "TRAINING": {
        "batch_size": 128,
        "patience": 20,
        "learning_rate": 0.01,
        "weight_decay": 1e-4,
    },
    "VQC": {
        "spsa_epsilon": 1e-6,
        "use_gpu": True,
    },
    "VERBOSE": True,
}

# Topologie da testare
TOPOLOGIES = ["circular", "ladder", "graph_ladder", "full"]

# Esperimenti: (d, readout, reps, spsa_batch_size, n_epochs)
EXPERIMENTS = [
    # --- Blocco A: baseline, scaling con d (readout fisso semplice) ---
    (4,  "x_z",              1, 1, 100),
    (8,  "x_z",              1, 1, 100),
    (16, "x_z",              1, 2, 100),
    # --- Blocco B: effetto reps a d fissato (isola la profondità ansatz) ---
    (16, "x_z",              2, 2, 100),
    # --- Blocco C: effetto readout a d, reps fissati ---
    (16, "x_y_z",            1, 2, 100),
    (16, "x_y_z_pair_nn",    1, 2, 100),
    (16, "x_y_z_pair_all",   1, 2, 100),
    # --- Blocco D: scaling a d più alto, miglior readout candidato (pair_nn) ---
    (32, "x_y_z_pair_nn",    2, 2, 200),
    (32, "x_y_z_pair_all",   2, 2, 200),
]

# ============================================
# ESPERIMENTI
# ============================================
print("\n⚠️  ATTENZIONE: Gli esperimenti richiederanno diverse ore.")
response = input("Procedere con gli esperimenti? (s/n): ")
if response.lower() != 's':
    print("Esperimenti annullati.")
    exit()

# Esecuzione
for topology in TOPOLOGIES:
    print(f"\n{'='*70}")
    print(f"🚀 TOPOLOGIA: {topology.upper()} (range [-π/2, π/2])")
    print(f"{'='*70}")

    for d, readout, reps, spsa_bs, n_epochs in EXPERIMENTS:
        tag = f"d{d}_{readout}_reps{reps}"

        save_dir = f'../artifacts/c3_pi_minus_half/{topology}'
        os.makedirs(save_dir, exist_ok=True)

        base_name = f'vqc_{tag}_topo_{topology}_seed_{SEED}'
        artifact_path = f'{save_dir}/{base_name}.json'

        if os.path.exists(artifact_path):
            print(f"[SKIP] {topology}/{tag} — già presente")
            continue

        print(f"\n--- {tag} (topology={topology}) ---")
        print(f"    Epoche: {n_epochs}, Batch size: {CFG['TRAINING']['batch_size']}")

        t_start = perf_counter()
        try:
            model, history, X_val, y_val, X_test, y_test = train_vqc(
                d=d, n_qubits=N_QUBITS, topology=topology,
                n_epochs=n_epochs,
                batch_size=CFG["TRAINING"]["batch_size"],
                patience=CFG["TRAINING"]["patience"],
                seed=SEED,
                spsa_epsilon=CFG["VQC"]["spsa_epsilon"],
                spsa_batch_size=spsa_bs,
                readout=readout,
                reps=reps,
                learning_rate=CFG["TRAINING"]["learning_rate"],
                weight_decay=CFG["TRAINING"]["weight_decay"]
            )

            val_metrics, _, _ = evaluate_vqc(model, X_val, y_val)
            test_metrics, _, _ = evaluate_vqc(model, X_test, y_test)
            elapsed_min = (perf_counter() - t_start) / 60

            # 🔴 FIX: readout/reps/spsa_bs/n_epochs passati esplicitamente,
            # non più ricavati dal tag dentro save_results
            save_results(
                model=model, d=d, n_qubits=N_QUBITS, topology=topology,
                readout=readout, reps=reps, spsa_batch_size=spsa_bs,
                seed=SEED, n_epochs_requested=n_epochs,
                history=history, val_metrics=val_metrics,
                test_metrics=test_metrics, elapsed_min=elapsed_min
            )

            print(f"\n✅ {tag} ({topology}):")
            print(f"   VAL  → F1={val_metrics['macro_f1']:.4f} | AUROC={val_metrics['macro_auroc']:.4f}")
            print(f"   TEST → F1={test_metrics['macro_f1']:.4f} | AUROC={test_metrics['macro_auroc']:.4f}")
            print(f"   Tempo: {elapsed_min:.1f} min")

        except Exception as e:
            print(f"❌ ERRORE {tag} ({topology}): {e}")
            import traceback
            traceback.print_exc()

print("\n" + "=" * 70)
print("🏁 TUTTI GLI ESPERIMENTI SONO COMPLETATI!")
print("=" * 70)


⚠️  ATTENZIONE: Gli esperimenti richiederanno diverse ore.

🚀 TOPOLOGIA: CIRCULAR (range [-π/2, π/2])

--- d4_x_z_reps1 (topology=circular) ---
    Epoche: 100, Batch size: 128
d= 4 | topology=circular | readout=x_z | var_params=16 | readout_dim=16
  GPU attiva: cuStateVec abilitato
  q_weights : torch.Size([16])
  head      : Linear(16, 4)
d=4 | topology=circular | epochs=100
--------------------------------------------------
epoch   1/100 | train_loss=1.3231 | val_loss=1.2787 | val_f1=0.3460 | val_auroc=0.7464 | val_bacc=0.4000 | val_ece=0.0594
epoch  11/100 | train_loss=1.0828 | val_loss=1.0941 | val_f1=0.5873 | val_auroc=0.8234 | val_bacc=0.5890 | val_ece=0.0948
epoch  21/100 | train_loss=1.0574 | val_loss=1.0823 | val_f1=0.5844 | val_auroc=0.8301 | val_bacc=0.5910 | val_ece=0.0681
epoch  31/100 | train_loss=1.0584 | val_loss=1.0794 | val_f1=0.5819 | val_auroc=0.8303 | val_bacc=0.5850 | val_ece=0.0583
Early stopping a epoch 36
Miglior val_f1: 0.5978 | val_auroc=0.8285 | val_bacc=0

In [1]:
import json
import glob

ARTIFACTS_ROOT = '../artifacts/c3_pi_minus_half'

# 👇 cambia in 'val' se vuoi ordinare per F1 di validation invece che di test
METRIC_SOURCE = 'test'  # 'test' oppure 'val'

# Solo le topologie reali: esclude eventuali cartelle di test (es. "test_run")
TOPOLOGIES = ["circular", "ladder", "graph_ladder", "full"]

artifact_files = glob.glob(f'{ARTIFACTS_ROOT}/*/*.json')

results = []
for path in artifact_files:
    with open(path, 'r') as f:
        r = json.load(f)
    if r.get('topology') in TOPOLOGIES:
        results.append(r)

if not results:
    print(f"⚠️  Nessun risultato trovato in {ARTIFACTS_ROOT}")
else:
    metrics_key = f'{METRIC_SOURCE}_metrics'

    # Ordina per macro F1, dal più alto al più basso
    results.sort(key=lambda r: r[metrics_key]['macro_f1'], reverse=True)

    print(f"\n{'='*105}")
    print(f"📊 RIEPILOGO ESPERIMENTI ({len(results)}/{len(TOPOLOGIES) * 9} completati) "
          f"— ordinati per {METRIC_SOURCE.upper()} F1 decrescente")
    print(f"{'='*105}")

    header = (f"{'#':>3} | {'topology':<13} | {'d':>4} | {'readout':<16} | {'reps':>4} | "
              f"{'val_f1':>7} | {'test_f1':>7} | {'test_auroc':>10} | {'test_bacc':>9} | "
              f"{'early_stop':>10} | {'min':>6}")
    print(header)
    print('-' * len(header))

    for i, r in enumerate(results, start=1):
        print(f"{i:>3} | {r['topology']:<13} | {r['d']:>4} | {r['readout']:<16} | {r['reps']:>4} | "
              f"{r['val_metrics']['macro_f1']:>7.4f} | {r['test_metrics']['macro_f1']:>7.4f} | "
              f"{r['test_metrics']['macro_auroc']:>10.4f} | {r['test_metrics']['bal_acc']:>9.4f} | "
              f"{str(r['early_stopped']):>10} | {r['elapsed_min']:>6.1f}")

    print(f"{'='*105}")
    best = results[0]
    print(f"🏆 Migliore ({METRIC_SOURCE} F1): topology={best['topology']} | d={best['d']} | "
          f"readout={best['readout']} | reps={best['reps']} → "
          f"test_f1={best['test_metrics']['macro_f1']:.4f} | val_f1={best['val_metrics']['macro_f1']:.4f}")


📊 RIEPILOGO ESPERIMENTI (36/36 completati) — ordinati per TEST F1 decrescente
  # | topology      |    d | readout          | reps |  val_f1 | test_f1 | test_auroc | test_bacc | early_stop |    min
-----------------------------------------------------------------------------------------------------------------------
  1 | circular      |   32 | x_y_z_pair_all   |    2 |  0.7253 |  0.7206 |     0.9073 |    0.7210 |       True |   34.7
  2 | ladder        |   32 | x_y_z_pair_all   |    2 |  0.7414 |  0.7177 |     0.9055 |    0.7180 |       True |   68.8
  3 | graph_ladder  |   32 | x_y_z_pair_all   |    2 |  0.7226 |  0.7096 |     0.9068 |    0.7080 |       True |   66.5
  4 | full          |   32 | x_y_z_pair_all   |    2 |  0.7257 |  0.6890 |     0.8864 |    0.6930 |       True |   95.3
  5 | full          |   32 | x_y_z_pair_nn    |    2 |  0.7164 |  0.6823 |     0.8803 |    0.6830 |       True |   66.5
  6 | circular      |   32 | x_y_z_pair_nn    |    2 |  0.6799 |  0.6597 |     0.